# Temporal Framing & Dataset Splits

Documents temporal structure: observation window, outcome window, and time-based splits.

Key constraint: no true timestamps. Using ID column as a temporal proxy—acknowledge the limitation upfront.

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset exactly as-is
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(list(df.columns))

## Schema Inspection

### Temporal/Sequential Columns

The dataset contains three groups of time-sequenced features:

1. **Payment Status (PAY_0 to PAY_6)**: Repayment status for months -1 to -6
   - PAY_0: Most recent month (month -1)
   - PAY_2 to PAY_6: Months -2 through -6 (historical)
   - Values: -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8
   - Interpretation: -2 = no consumption, -1 = pay duly, 0 = use of revolving credit, 1+ = delay in months

2. **Bill Amounts (BILL_AMT1 to BILL_AMT6)**: Statement bill amounts
   - BILL_AMT1: Most recent month (month -1)
   - BILL_AMT2 to BILL_AMT6: Months -2 through -6

3. **Payment Amounts (PAY_AMT1 to PAY_AMT6)**: Previous payment amounts
   - PAY_AMT1: Most recent month (month -1)
   - PAY_AMT2 to PAY_AMT6: Months -2 through -6

### Static Features

- **ID**: Unique identifier (1 to 30000)
- **LIMIT_BAL**: Credit limit
- **SEX**: Gender (1=male, 2=female)
- **EDUCATION**: Education level (1-4)
- **MARRIAGE**: Marital status (1-3)
- **AGE**: Age in years

### Target Variable

- **default payment next month**: Binary target (0=no default, 1=default)
  - This is the outcome for month 0 (the month following the most recent observation)

In [ ]:
# Inspect temporal columns
temporal_cols = {
    'payment_status': [f'PAY_{i}' if i > 0 else 'PAY_0' for i in [0, 2, 3, 4, 5, 6]],
    'bill_amounts': [f'BILL_AMT{i}' for i in range(1, 7)],
    'payment_amounts': [f'PAY_AMT{i}' for i in range(1, 7)]
}

print("Temporal column groups:")
for group_name, cols in temporal_cols.items():
    print(f"\n{group_name.upper()}:")
    print(f"  Columns: {cols}")
    if group_name == 'payment_status':
        print(f"  Unique values in PAY_0: {sorted(df['PAY_0'].unique())}")

## Temporal Framing

### Observation Window (Information Available at Decision Time)

What we know when making a decision: months -6 to -1.

Historical payment behavior (PAY_6 through PAY_0), bill amounts (BILL_AMT6 through BILL_AMT1), payment amounts (PAY_AMT6 through PAY_AMT1). Plus static features: credit limit, demographics, age.

Decision happens at end of month -1 using months -6 to -1.

### Outcome Window (Default Definition)

What we observe: month 0 only.

Target: `default payment next month` = 1 if default in month 0, 0 otherwise.

Default means: failure to make minimum payment (dataset definition).

Limitation: single-month observation. Defaults after month 0 aren't captured—creates label noise from delayed defaults.

## Temporal Split Logic

### Critical Constraint: No True Timestamps

Problem: no observation dates. Can't split by calendar time.

Solution: use ID column as temporal proxy. Not perfect, but better than random splits.

### Proxy Ordering: ID Column

Assumption: lower IDs = earlier observations, higher IDs = later observations.

Why this works: credit datasets usually add records sequentially. ID correlates with time. Common practice when timestamps are missing.

Limitation: if IDs were random, this fails. Acknowledging the assumption and proceeding.

### Split Strategy

Train: IDs 1-20,000 (~67%)
Val: IDs 20,001-25,000 (~17%)
Test: IDs 25,001-30,000 (~17%)

Why this split: mimics production (train on history, test on future). Prevents temporal leakage. Exposes distribution shift.

In [ ]:
# Define temporal splits using ID as proxy
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

print(f"Train set: {len(train_ids)} samples (IDs {train_ids.min()} to {train_ids.max()})")
print(f"Validation set: {len(val_ids)} samples (IDs {val_ids.min()} to {val_ids.max()})")
print(f"Test set: {len(test_ids)} samples (IDs {test_ids.min()} to {test_ids.max()})")

# Verify no overlap
assert len(set(train_ids) & set(val_ids)) == 0, "Train/Val overlap!"
assert len(set(train_ids) & set(test_ids)) == 0, "Train/Test overlap!"
assert len(set(val_ids) & set(test_ids)) == 0, "Val/Test overlap!"
print("\n✓ No overlap between splits")

## Assumptions & Limitations

### What This Temporal Proxy Captures

Sequential ordering (lower IDs = earlier). Temporal validation (test = "future"). Distribution shift exposure (performance may degrade). No future leakage (strict ordering enforced).

### What It Cannot Capture

True calendar time (don't know dates—were IDs 1-1000 from 2020 or 2023?). Seasonal effects. Economic cycles. Cohort effects. Gap periods (if IDs have gaps, proxy fails).

### Why It's Still Acceptable

Standard practice in credit modeling when timestamps are missing. More realistic than random splits (which leak future information). Explicitly documented assumptions enable proper interpretation. Sufficient for this project's scope: prevent leakage, expose shift, validate stability.

Drift detection project handles true temporal monitoring in production.

### Key Takeaway

This is a necessary approximation. Results should account for unknown temporal relationships. Split prioritizes preventing leakage and exposing degradation over perfect temporal alignment.